# RAG Implementations: Simple → Pro

This notebook walks through **two complete RAG pipelines** for the Insurellm Expert Assistant:

| | Implementation | Pro Implementation |
|---|---|---|
| **Chunking** | Fixed-size (LangChain splitter) | LLM-powered smart chunks |
| **Retrieval** | Single vector search | Dual search (original + rewritten query) |
| **Re-ranking** | ❌ | ✅ LLM re-ranks results |
| **Query rewriting** | ❌ | ✅ LLM rewrites query from context |
| **Library** | LangChain | Raw OpenAI SDK + litellm |

---

# 🟢 PART 1: Simple Implementation

A clean, LangChain-based RAG pipeline — straightforward and easy to understand.

### Flow:
```
Documents → Fixed-size chunks → OpenAI Embeddings → Chroma VectorDB
Question  → Retrieve top-k chunks → Inject into System Prompt → LLM Answer
```

## Step 1.1 — Imports & Config

In [2]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

load_dotenv(override=True)

MODEL          = "gpt-4.1-nano"
DB_NAME        = str(Path("..").resolve() / "vector_db")
KNOWLEDGE_BASE = str(Path("..").resolve() / "knowledge-base")
RETRIEVAL_K    = 10

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print("✅ Config ready")

✅ Config ready


## Step 1.2 — Load Documents from Knowledge Base

Each subfolder in `knowledge-base/` is a **document type** (e.g. `employees`, `products`, `contracts`).  
We load all `.md` files and tag each document with its `doc_type`.

In [3]:
def fetch_documents():
    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder,
            glob="**/*.md",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

documents = fetch_documents()
print(f"📄 Loaded {len(documents)} documents")
print(f"   Types: {set(d.metadata['doc_type'] for d in documents)}")

# Preview first document
print(f"\n--- First document preview ---")
print(f"Source: {documents[0].metadata['source']}")
print(documents[0].page_content[:300])

📄 Loaded 76 documents
   Types: {'company', 'products', 'employees', 'contracts'}

--- First document preview ---
Source: /Users/bahlu/Desktop/Bahlu/learning/llm_engineering/week5/knowledge-base/products/Rellm.md
# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced pla


## Step 1.3 — Split into Chunks

`RecursiveCharacterTextSplitter` splits on natural boundaries (paragraphs → sentences → words).  
- `chunk_size=500` — max characters per chunk  
- `chunk_overlap=200` — overlap so context is not lost at boundaries

In [4]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    return text_splitter.split_documents(documents)

chunks = create_chunks(documents)
print(f"✂️  Split into {len(chunks)} chunks")
print(f"\n--- Example chunk ---")
print(chunks[0].page_content)

✂️  Split into 970 chunks

--- Example chunk ---
# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary


## Step 1.4 — Embed & Store in Chroma Vector DB

Each chunk is converted into a **1536-dim embedding vector** using `text-embedding-3-large`  
and stored in a local **Chroma** vector database for fast similarity search.

In [ ]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )
    collection = vectorstore._collection
    count = collection.count()
    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"🗄️  VectorDB ready: {count:,} vectors with {dimensions:,} dimensions")
    return vectorstore

# ⚠️ This makes API calls — skip if vector_db already exists
vectorstore_simple = create_embeddings(chunks)
print("✅ Ingestion complete")

## Step 1.5 — Set Up Retriever & LLM

In [ ]:
# Load existing vectorstore (skip re-ingestion if already done)
vectorstore_simple = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

retriever = vectorstore_simple.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

print("🔍 Retriever ready")
print("🤖 LLM ready")

## Step 1.6 — Test Retrieval

Verify the retriever finds relevant chunks for a test question.

In [ ]:
test_question = "Who is Avery Lancaster?"
retrieved_docs = retriever.invoke(test_question, k=RETRIEVAL_K)

print(f"🔍 Retrieved {len(retrieved_docs)} docs for: '{test_question}'\n")
for i, doc in enumerate(retrieved_docs[:3], 1):
    print(f"--- Doc {i} | source: {doc.metadata['source']} ---")
    print(doc.page_content[:200])
    print()

## Step 1.7 — System Prompt & Answer Function

The retrieved chunks are injected into the system prompt as `{context}`.  
We also combine prior conversation history to make the assistant context-aware.

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def fetch_context(question: str) -> list[Document]:
    """Retrieve relevant context documents for a question."""
    return retriever.invoke(question, k=RETRIEVAL_K)

def combined_question(question: str, history: list[dict] = []) -> str:
    """Combine all user messages from history + current question for better retrieval."""
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """Answer the given question with RAG; return the answer and the context documents."""
    combined = combined_question(question, history)
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs

print("✅ Answer function ready")

## Step 1.8 — Test the Simple RAG Pipeline

In [ ]:
question = "Who is Avery Lancaster?"
answer, context_docs = answer_question(question)

print(f"❓ Question: {question}\n")
print(f"💬 Answer:\n{answer}\n")
print(f"📚 Context sources used:")
for doc in context_docs:
    print(f"   - {doc.metadata['source']}")

In [ ]:
# Try another question with history (multi-turn)
history = [
    {"role": "user", "content": "Who is Avery Lancaster?"},
    {"role": "assistant", "content": answer}
]

followup = "What products did she launch?"
answer2, _ = answer_question(followup, history)

print(f"❓ Follow-up: {followup}\n")
print(f"💬 Answer:\n{answer2}")

## Step 1.9 — Launch Gradio UI (Simple Implementation)

In [ ]:
import gradio as gr

def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result

def chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Insurellm Expert Assistant (Simple)", theme=theme) as ui_simple:
    gr.Markdown("# 🏢 Insurellm Expert Assistant — Simple Implementation")
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="💬 Conversation", height=500, type="messages")
            message = gr.Textbox(placeholder="Ask anything about Insurellm...", show_label=False)
        with gr.Column(scale=1):
            context_md = gr.Markdown(
                value="*Retrieved context will appear here*",
                container=True,
                height=500,
            )
    message.submit(
        put_message_in_chatbot, [message, chatbot], [message, chatbot]
    ).then(chat, chatbot, [chatbot, context_md])

ui_simple.launch(inbrowser=True)

---

# 🔴 PART 2: Pro Implementation

A production-grade RAG pipeline with **LLM-powered chunking**, **query rewriting**, **dual retrieval**, and **LLM re-ranking**.

### Flow:
```
INGEST:
  Documents → LLM splits into smart chunks (headline + summary + original_text)
           → OpenAI Embeddings → Raw ChromaDB (preprocessed_db)

ANSWER:
  Question → LLM rewrites query (context-aware)
          → Dual vector search (original + rewritten)
          → Merge results
          → LLM re-ranks by relevance
          → Top-K chunks → System Prompt → LLM Answer
```

## Step 2.1 — Imports & Config

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from litellm import completion
from tqdm import tqdm
from multiprocessing import Pool
from tenacity import retry, wait_exponential
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)

# MODEL = "openai/gpt-4.1-nano"   # cheaper option
MODEL              = "groq/openai/gpt-oss-120b"  # more capable
PRO_DB_NAME        = str(Path("..").resolve() / "preprocessed_db")
KNOWLEDGE_BASE_PATH = Path("..").resolve() / "knowledge-base"
COLLECTION_NAME    = "docs"
EMBEDDING_MODEL    = "text-embedding-3-large"
AVERAGE_CHUNK_SIZE = 100   # used to estimate how many chunks to create per doc
WORKERS            = 3     # parallel workers for LLM chunking (set to 1 if rate limited)
RETRIEVAL_K        = 20    # how many chunks to fetch from DB
FINAL_K            = 10    # how many to keep after re-ranking
wait               = wait_exponential(multiplier=1, min=10, max=240)

openai_client = OpenAI()
print("✅ Pro config ready")

## Step 2.2 — Data Models (Pydantic)

We use Pydantic models to enforce **structured output** from the LLM.

Each `Chunk` produced by the LLM has:
- **`headline`** — a short label (what queries might match this)
- **`summary`** — a few sentences about the chunk's content  
- **`original_text`** — the raw text from the document, unchanged

This is stored as `headline + summary + original_text` — richer than raw text alone.

In [ ]:
class Result(BaseModel):
    """A retrieved chunk with its metadata."""
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    """A single LLM-generated chunk with headline, summary, and original text."""
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        """Convert to a Result combining headline + summary + original_text."""
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
            metadata=metadata,
        )

class Chunks(BaseModel):
    """A list of chunks returned by the LLM."""
    chunks: list[Chunk]

class RankOrder(BaseModel):
    """Re-ranked order of chunk IDs from most to least relevant."""
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

print("✅ Pydantic models defined")

## Step 2.3 — Load Documents (Homemade Loader)

Instead of LangChain's `DirectoryLoader`, we use a simple homemade loader.

In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader."""
    documents = []
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({
                    "type": doc_type,
                    "source": file.as_posix(),
                    "text": f.read()
                })
    print(f"📄 Loaded {len(documents)} documents")
    return documents

pro_documents = fetch_documents()

# Preview
print(f"\n--- First document ---")
print(f"Type:   {pro_documents[0]['type']}")
print(f"Source: {pro_documents[0]['source']}")
print(pro_documents[0]['text'][:300])

## Step 2.4 — LLM-Powered Smart Chunking

Instead of blindly splitting by character count, we ask the **LLM itself** to read each document and produce semantically meaningful chunks with headlines and summaries.

The prompt estimates how many chunks are needed based on document length.

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned across the chunks - don't leave anything out.
This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate, ensuring that there are individual chunks to answer specific questions.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

def make_messages(document):
    return [{"role": "user", "content": make_prompt(document)}]

print("✅ Prompt builder ready")

# Preview the prompt for one document
print("\n--- Prompt preview (first 500 chars) ---")
print(make_prompt(pro_documents[0])[:500])

In [ ]:
@retry(wait=wait)
def process_document(document):
    """Ask the LLM to chunk one document into structured Chunk objects."""
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

# Test on a single document first
print("🧪 Testing LLM chunking on one document...")
test_chunks = process_document(pro_documents[0])
print(f"✅ Got {len(test_chunks)} chunks from the LLM\n")
print("--- First chunk ---")
print(test_chunks[0].page_content[:400])

In [ ]:
def create_chunks(documents):
    """
    Create smart chunks using parallel workers.
    If you get rate limit errors, set WORKERS = 1 above.
    """
    chunks = []
    with Pool(processes=WORKERS) as pool:
        for result in tqdm(pool.imap_unordered(process_document, documents), total=len(documents)):
            chunks.extend(result)
    return chunks

# ⚠️ This makes many API calls (one per document) — skip if preprocessed_db already exists
print("🔄 Creating smart chunks for all documents (this may take a few minutes)...")
pro_chunks = create_chunks(pro_documents)
print(f"✅ Total chunks created: {len(pro_chunks)}")

## Step 2.5 — Embed & Store in Raw Chroma DB

We use the raw `chromadb` client (no LangChain wrapper) for more control.

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=PRO_DB_NAME)

    # Clear existing collection if present
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)

    texts   = [chunk.page_content for chunk in chunks]
    metas   = [chunk.metadata for chunk in chunks]

    # Batch embed all chunks
    print("🔢 Generating embeddings...")
    emb     = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"🗄️  Pro VectorDB ready: {collection.count()} documents stored")

# ⚠️ Skip if preprocessed_db already exists
create_embeddings(pro_chunks)
print("✅ Pro ingestion complete")

## Step 2.6 — Connect to Pro VectorDB

In [ ]:
pro_chroma     = PersistentClient(path=PRO_DB_NAME)
pro_collection = pro_chroma.get_or_create_collection(COLLECTION_NAME)

print(f"🗄️  Connected to Pro VectorDB: {pro_collection.count()} chunks available")

## Step 2.7 — Query Rewriting

Before searching, the LLM **rewrites the user's question** into a more specific, context-aware query.  
This handles follow-up questions like "What role does she have?" (who is "she"?)

In [ ]:
@retry(wait=wait)
def rewrite_query(question, history=None):
    """Rewrite the user's question to be more specific and context-aware."""
    if history is None:
        history = []

    message = f"""
You are in a conversation with a user.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Since the conversation is contextual, understand the meaning of the user question and add details based on the history.
Condense everything in a single contextually-rich VERY short and specific question, most likely to surface content.

EXAMPLE:
user: Who is the founder? -> Query: who is the founder?
assistant: The founder is FooBar
user: What role covers? -> Query: What role FooBar covers?
...

IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

# Test query rewriting
original = "What about her products?"
history_example = [
    {"role": "user", "content": "Who is Avery Lancaster?"},
    {"role": "assistant", "content": "Avery Lancaster is the Co-Founder and CEO of Insurellm."}
]
rewritten = rewrite_query(original, history_example)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")

## Step 2.8 — Dual Retrieval + Merge

We run **two** vector searches — one with the original question, one with the rewritten query — then merge unique results.

In [ ]:
def fetch_context_unranked(question):
    """Run a single vector search and return Result objects."""
    query_vec = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=[question]
    ).data[0].embedding

    results = pro_collection.query(query_embeddings=[query_vec], n_results=RETRIEVAL_K)
    chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks

def merge_chunks(chunks, reranked):
    """Merge two result lists, keeping unique chunks only."""
    merged = chunks[:]
    existing = [chunk.page_content for chunk in chunks]
    for chunk in reranked:
        if chunk.page_content not in existing:
            merged.append(chunk)
    return merged

# Test dual retrieval
q = "Who is Avery Lancaster?"
chunks1 = fetch_context_unranked(q)
chunks2 = fetch_context_unranked(rewrite_query(q))
merged  = merge_chunks(chunks1, chunks2)

print(f"Original query results : {len(chunks1)}")
print(f"Rewritten query results: {len(chunks2)}")
print(f"Merged (unique)        : {len(merged)}")

## Step 2.9 — LLM Re-ranking

After retrieval, we ask the LLM to **re-order** the chunks by relevance to the question.  
This goes beyond pure vector similarity — the LLM understands the question semantically.

In [ ]:
@retry(wait=wait)
def rerank(question, chunks):
    """Ask the LLM to re-rank chunks by relevance to the question."""
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply    = response.choices[0].message.content
    order    = RankOrder.model_validate_json(reply).order
    return [chunks[i - 1] for i in order]

# Test re-ranking
reranked = rerank(q, merged)
final    = reranked[:FINAL_K]
print(f"After re-ranking, top {FINAL_K} sources:")
for c in final:
    print(f"  - {c.metadata['source']}")

## Step 2.10 — Full `fetch_context` Pipeline

Combines query rewriting → dual retrieval → merge → re-rank → top-K.

In [ ]:
def fetch_context(original_question, history=None):
    """Full pro retrieval pipeline."""
    rewritten_question = rewrite_query(original_question, history)
    print(f"  📝 Rewritten query: {rewritten_question}")

    chunks1 = fetch_context_unranked(original_question)
    chunks2 = fetch_context_unranked(rewritten_question)
    chunks  = merge_chunks(chunks1, chunks2)
    print(f"  🔍 Retrieved {len(chunks)} unique chunks (dual search)")

    reranked = rerank(original_question, chunks)
    top_k    = reranked[:FINAL_K]
    print(f"  🏆 Re-ranked, keeping top {FINAL_K}")
    return top_k

print("✅ fetch_context pipeline ready")

## Step 2.11 — Pro Answer Function

In [ ]:
PRO_SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks
    )
    system_prompt = PRO_SYSTEM_PROMPT.format(context=context)
    return (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )

@retry(wait=wait)
def answer_question_pro(question: str, history: list[dict] = []) -> tuple[str, list]:
    """Answer a question using the full Pro RAG pipeline."""
    chunks   = fetch_context(question, history)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

print("✅ Pro answer function ready")

## Step 2.12 — Test the Pro RAG Pipeline

In [ ]:
question = "Who is Avery Lancaster?"

print(f"❓ Question: {question}\n")
answer_pro, context_chunks = answer_question_pro(question)

print(f"\n💬 Answer:\n{answer_pro}\n")
print(f"📚 Context sources used:")
for chunk in context_chunks:
    print(f"   - {chunk.metadata['source']}")

In [ ]:
# Multi-turn test — follow-up question
history = [
    {"role": "user",      "content": "Who is Avery Lancaster?"},
    {"role": "assistant", "content": answer_pro}
]

followup = "What products did she launch?"
print(f"❓ Follow-up: {followup}\n")
answer_pro2, _ = answer_question_pro(followup, history)
print(f"💬 Answer:\n{answer_pro2}")

## Step 2.13 — Launch Gradio UI (Pro Implementation)

In [ ]:
import gradio as gr

def format_context_pro(chunks):
    result = "<h2 style='color: #6200ea;'>Relevant Context (Re-ranked)</h2>\n\n"
    for i, chunk in enumerate(chunks, 1):
        result += f"<span style='color: #6200ea;'>#{i} Source: {chunk.metadata['source']}</span>\n\n"
        result += chunk.page_content + "\n\n"
    return result

def chat_pro(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, chunks = answer_question_pro(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context_pro(chunks)

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Insurellm Expert Assistant (Pro)", theme=theme) as ui_pro:
    gr.Markdown("# 🏢 Insurellm Expert Assistant — Pro Implementation\n*With query rewriting, dual retrieval & LLM re-ranking*")
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="💬 Conversation", height=500, type="messages")
            message = gr.Textbox(placeholder="Ask anything about Insurellm...", show_label=False)
        with gr.Column(scale=1):
            context_md = gr.Markdown(
                value="*Retrieved & re-ranked context will appear here*",
                container=True,
                height=500,
            )
    message.submit(
        put_message_in_chatbot, [message, chatbot], [message, chatbot]
    ).then(chat_pro, chatbot, [chatbot, context_md])

ui_pro.launch(inbrowser=True)

---

# 📊 Summary: Simple vs Pro

| Step | Simple Implementation | Pro Implementation |
|------|----------------------|--------------------|
| **Document loading** | LangChain `DirectoryLoader` | Homemade loader |
| **Chunking** | Fixed-size (`RecursiveCharacterTextSplitter`) | LLM-generated (headline + summary + text) |
| **Chunk quality** | Raw text only | Rich: headline + summary + original |
| **Parallelism** | Single-threaded | Multiprocessing (3 workers) |
| **Vector DB** | LangChain `Chroma` wrapper | Raw `chromadb.PersistentClient` |
| **Query** | Raw user question | LLM rewrites question first |
| **Retrieval** | One vector search | Two searches (original + rewritten) |
| **Result merging** | N/A | Merges unique chunks from both searches |
| **Re-ranking** | ❌ | ✅ LLM re-ranks by relevance |
| **Retry logic** | ❌ | ✅ Exponential backoff on all LLM calls |
| **Model** | `gpt-4.1-nano` | `gpt-oss-120b` via litellm |
| **Best for** | Learning, prototyping | Production use |
